<a href="https://colab.research.google.com/github/jannellemagdasal/Building-a-Custom-Image-Classifier-with-TensorFlow/blob/main/lw5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
dataset_path = "/content/drive/MyDrive/ImageDataset"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import tensorflow as tf
img_height, img_width = 224, 224
batch_size = 32
train_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_path,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)
val_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_path,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)
class_names = train_ds.class_names

Found 9593 files belonging to 20 classes.
Using 7675 files for training.
Found 9593 files belonging to 20 classes.
Using 1918 files for validation.


In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

def build_model(base_model):
    base_model.trainable = False  # Transfer Learning

    model = models.Sequential([
        layers.Input(shape=(224, 224, 3)),  # FIXED warning here
        layers.Rescaling(1./255),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(len(class_names))
    ])

    return model

# Create base model
base_model = MobileNetV2(weights='imagenet', include_top=False)

# Build model
model = build_model(base_model)

/tmp/ipykernel_11032/1679439141.py:20: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False)


In [ ]:
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, DenseNet121

models_dict = {
    "MobileNetV2": MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3)),
    "EfficientNetB0": EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3)),
    "DenseNet121": DenseNet121(weights='imagenet', include_top=False, input_shape=(224,224,3))
}

In [ ]:
from tensorflow.keras.optimizers import Adam
histories = {}
trained_models = {}
for name, base in models_dict.items():
  print(f"Training {name}...")
  model = build_model(base)
  model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
  history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)
histories[name] = history
trained_models[name] = model

Training MobileNetV2...
Epoch 1/10
240/240 ━━━━━━━━━━━━━━━━━━━━ 537s 2s/step - accuracy: 0.2474 - loss: 2.5229 - val_accuracy: 0.6653 - val_loss: 1.5650
Epoch 2/10
240/240 ━━━━━━━━━━━━━━━━━━━━ 463s 2s/step - accuracy: 0.5696 - loss: 1.4595 - val_accuracy: 0.8321 - val_loss: 0.8360
Epoch 3/10
240/240 ━━━━━━━━━━━━━━━━━━━━ 567s 2s/step - accuracy: 0.7207 - loss: 0.9765 - val_accuracy: 0.8999 - val_loss: 0.5311
Epoch 4/10
240/240 ━━━━━━━━━━━━━━━━━━━━ 425s 2s/step - accuracy: 0.7953 - loss: 0.7111 - val_accuracy: 0.9275 - val_loss: 0.3841
Epoch 5/10
240/240 ━━━━━━━━━━━━━━━━━━━━ 475s 2s/step - accuracy: 0.8392 - loss: 0.5702 - val_accuracy: 0.9426 - val_loss: 0.2948
Epoch 6/10
240/240 ━━━━━━━━━━━━━━━━━━━━ 414s 2s/step - accuracy: 0.8756 - loss: 0.4603 - val_accuracy: 0.9510 - val_loss: 0.2375
Epoch 7/10
240/240 ━━━━━━━━━━━━━━━━━━━━ 515s 2s/step - accuracy: 0.8903 - loss: 0.3910 - val_accuracy: 0.9640 - val_loss: 0.1956
Epoch 8/10
240/240 ━━━━━━━━━━━━━━━━━━━━ 478s 2s/step - accuracy: 0.9161 -

In [ ]:
from sklearn.metrics import classification_report

def evaluate_model(model):
    y_true, y_pred = [], []

    for images, labels in val_ds:
        preds = model(images, training=False)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.numpy().argmax(axis=1))

    print(classification_report(y_true, y_pred))

    return y_true, y_pred

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure()
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()

    plt.xticks(np.arange(len(class_names)), class_names, rotation=45)
    plt.yticks(np.arange(len(class_names)), class_names)

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j, i, cm[i, j], ha='center')

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.preprocessing import label_binarize
import numpy as np
import matplotlib.pyplot as plt

def plot_roc(model):
    y_true, y_prob = [], []

    for images, labels in val_ds:
        preds = model(images, training=False)
        y_true.extend(labels.numpy())
        y_prob.extend(preds.numpy())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    y_bin = label_binarize(y_true, classes=range(len(class_names)))

    plt.figure()

    for i in range(len(class_names)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        plt.plot(fpr, tpr, label=class_names[i])

    plt.title("ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()

In [ ]:
print("Testing val_ds...")

count = 0
for images, labels in val_ds.take(1):
    print("Images shape:", images.shape)
    print("Labels:", labels.numpy())
    count += 1

print("Batches found:", count)

Testing val_ds...


NameError: name 'val_ds' is not defined

In [ ]:
for images, labels in val_ds.take(1):
    preds = model(images, training=False)
    print("Predictions shape:", preds.shape)

NameError: name 'val_ds' is not defined

In [ ]:
def evaluate_model(model):
    y_true, y_pred = [], []

    print("Starting evaluation...")

    for i, (images, labels) in enumerate(val_ds):
        print(f"Processing batch {i}")  # DEBUG

        preds = model(images, training=False)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.numpy().argmax(axis=1))

    print("Finished loop")
    print("Total samples:", len(y_true))

    print(classification_report(y_true, y_pred))

    return y_true, y_pred

In [ ]:
print("Train batches:", len(train_ds))
print("Validation batches:", len(val_ds))

NameError: name 'train_ds' is not defined

In [ ]:
def evaluate_model(model):
    y_true, y_pred = [], []

    print("Starting evaluation...")

    for i, (images, labels) in enumerate(val_ds):
        if i % 10 == 0:
            print(f"Processing batch {i}/{len(val_ds)}")  # 👈 progress

        preds = model(images, training=False)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.numpy().argmax(axis=1))

    print("Finished evaluation")
    print("Total samples:", len(y_true))

    from sklearn.metrics import classification_report
    print(classification_report(y_true, y_pred))

    return y_true, y_pred

In [ ]:
def evaluate_model(model):
    print("Running full prediction...")

    y_prob = model.predict(val_ds, verbose=1)  # 👈 progress bar built-in
    y_pred = y_prob.argmax(axis=1)

    y_true = np.concatenate([y for x, y in val_ds], axis=0)

    from sklearn.metrics import classification_report
    print(classification_report(y_true, y_pred))

    return y_true, y_pred

In [ ]:
y_true, y_pred = evaluate_model(model)

plot_confusion_matrix(y_true, y_pred, "Confusion Matrix")

plot_roc(model)

NameError: name 'model' is not defined

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    import matplotlib.pyplot as plt
    import numpy as np
    from sklearn.metrics import confusion_matrix

    print("Plotting confusion matrix...")  # 👈 debug

    cm = confusion_matrix(y_true, y_pred)

    # Normalize safely
    cm = cm.astype(float)
    cm = np.divide(cm, cm.sum(axis=1, keepdims=True),
                   where=cm.sum(axis=1, keepdims=True)!=0)

    plt.figure(figsize=(14, 12))
    plt.imshow(cm)

    plt.title(title)
    plt.colorbar()

    # 👇 Use numeric labels FIRST (guaranteed to work)
    num_classes = cm.shape[0]
    plt.xticks(np.arange(num_classes), np.arange(num_classes), rotation=90)
    plt.yticks(np.arange(num_classes), np.arange(num_classes))

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.tight_layout()
    plt.show()

In [ ]:
y_true, y_pred = evaluate_model(model)
plot_confusion_matrix(y_true, y_pred, "Confusion Matrix")

In [ ]:
print(len(class_names))  # should be 20
print(len(set(y_true)))  # should also be 20

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import load_img, img_to_array
img_path = "/content/drive/MyDrive/test_image.jpg"
img = load_img(img_path, target_size=(180, 180))
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

In [ ]:
for layer in model.layers:
  print(layer.name)

In [ ]:
last_conv_layer_name = "dense_7" # Adjust based on your model

In [ ]:
import tensorflow as tf
import numpy as np
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
  grad_model = tf.keras.models.Model(
    [model.inputs],
    [model.get_layer(last_conv_layer_name).output, model.output]
  )
  with tf.GradientTape() as tape:
    conv_outputs, predictions = grad_model(img_array)
    if pred_index is None:
      pred_index = tf.argmax(predictions[0])
    class_channel = predictions[:, pred_index]
  grads = tape.gradient(class_channel, conv_outputs)
  pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
  conv_outputs = conv_outputs[0]
  heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
  heatmap = tf.squeeze(heatmap)
  heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
  return heatmap.numpy()

In [ ]:
for i, layer in enumerate(model.layers):
    print(i, layer.name, layer.__class__.__name__)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Ensure correct input shape
img_array = tf.convert_to_tensor(img_array)
if len(img_array.shape) == 5:
    img_array = tf.squeeze(img_array, axis=0)
if len(img_array.shape) == 3:
    img_array = tf.expand_dims(img_array, axis=0)

img_array = tf.image.resize(img_array, (224, 224))

# Build the model once
_ = model(img_array, training=False)

# Get the last conv layer
last_conv_layer = model.get_layer(last_conv_layer_name)

# Create a model that returns the conv output and final prediction
grad_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[last_conv_layer.output, model.outputs]
)

with tf.GradientTape() as tape:
    conv_outputs, predictions = grad_model(img_array)
    class_idx = tf.argmax(predictions[0])
    class_channel = predictions[:, class_idx]

grads = tape.gradient(class_channel, conv_outputs)
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

conv_outputs = conv_outputs[0]
heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

heatmap = tf.maximum(heatmap, 0)
heatmap /= tf.reduce_max(heatmap)

plt.matshow(heatmap)
plt.title("Grad-CAM Heatmap")
plt.colorbar()
plt.show()

In [ ]:
for layer in model.layers:
    print(layer.name, layer.__class__.__name__)

In [ ]:
print(img_array.shape)

In [ ]:
model.save("/content/drive/MyDrive/model_name.keras")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import load_img, img_to_array
img_path = "/content/drive/MyDrive/test_image.jpg"
img = load_img(img_path, target_size=(240, 240))
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

In [ ]:
import requests
from PIL import Image
from io import BytesIO

# URL of a sample image (e.g., from Unsplash or a similar public domain source)
sample_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/DALL-E_image_of_a_cat_wearing_a_cowboy_hat.jpg/640px-DALL-E_image_of_a_cat_wearing_a_cowboy_hat.jpg"

# Path to save the downloaded image in Google Drive
save_path = "/content/drive/MyDrive/test_image.jpg"

try:
    response = requests.get(sample_image_url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    img = Image.open(BytesIO(response.content))
    # Resize if needed to ensure it works well with the model input size (e.g., 224x224 or 240x240)
    # img = img.resize((240, 240))
    img.save(save_path)
    print(f"Sample image downloaded and saved to: {save_path}")

except requests.exceptions.RequestException as e:
    print(f"Error downloading image: {e}")
except Exception as e:
    print(f"Error processing image: {e}")


In [ ]:
y_true, y_pred = evaluate_model(model)
plot_confusion_matrix(y_true, y_pred, "Confusion Matrix")

NameError: name 'evaluate_model' is not defined